# ResNet-56 on CIFAR-10

ResNet-56 takes a $3\times32\times32$ RGB image and returns ten class scores. Its main building block is a pair of convolutions with a shortcut around them. This notebook builds the CIFAR-10 architecture from the original paper and runs a forward pass. [Deep Residual Learning for Image Recognition](https://arxiv.org/pdf/1512.03385)

## 1. Why residual connections

In Figure 1, the 56-layer plain CNN has higher **training error** than the 20-layer plain CNN. The deeper network has enough layers to reproduce the shallower network in principle: copy the original layers and make the extra layers pass their input through unchanged. Finding weights that give this behavior is an optimization problem. [Paper introduction and Figure 1](https://arxiv.org/pdf/1512.03385#page=1)

A residual block gives the input $x$ a shortcut and lets two convolutions learn a correction $F(x)$. For a same-shape block, the two branches are added and then passed through ReLU:

$$y = \operatorname{ReLU}(F(x) + x).$$

If the input already contains useful features, a small $F(x)$ lets the block keep most of those features. [ResNet paper, Figure 2](https://arxiv.org/pdf/1512.03385#page=2)

### ResNet and Highway Networks

Both architectures give the input a short path through the block. For blocks that keep the same shape, the difference is how they combine that path with the transformed features.

In a **Highway block**, $H(x)$ is the transformed input and $T(x)$ is a learned sigmoid gate. The gate weights the transformed features, and its complement $1-T(x)$ weights the original input. Here $\odot$ means element-wise multiplication.

$$y = T(x)\odot H(x) + (1-T(x))\odot x = x + T(x)\odot(H(x)-x).$$

This form shows that Highway learns both the transformation and how much of its change to apply. When the gate is close to zero, the output stays close to $x$. [Highway Networks paper, Section 2](https://arxiv.org/pdf/1505.00387#page=2) · [Highway Networks exploration](../../highway-networks/notebooks/02_paper_exploration.ipynb)

In a **ResNet block**, the identity shortcut contributes the full $x$ to the sum before ReLU. The residual branch learns the correction $F(x)$ directly. A small residual keeps the sum close to the input, and the identity shortcut adds zero learned parameters. [ResNet paper, Sections 2 and 3.2](https://arxiv.org/pdf/1512.03385#page=2)

## 2. The residual block

The **shortcut** is a second path from the block’s input to the addition near its output. The input travels down both paths:

- **Residual branch.** A $3\times3$ convolution, batch normalization, and ReLU are followed by another $3\times3$ convolution and batch normalization. This path learns the correction $F(x)$.
- **Shortcut branch.** In a same-shape block, the input bypasses both convolutions and arrives unchanged. This is the **identity shortcut** and has no learned parameters.

The addition combines values at matching batch, channel, row, and column indices. For example, if two shortcut values are $[2, 1]$ and the learned corrections are $[-0.5, -1.5]$, their sum is $[1.5, -0.5]$. The final ReLU gives $[1.5, 0]$. During backpropagation, where this ReLU is active, the identity shortcut also gives the gradient a direct path back to the block’s input. [Paper, Section 3.2 and Figure 2](https://arxiv.org/pdf/1512.03385#page=3)

### Shortcut options A, B, and C

A **projection shortcut** is a learned $1\times1$ convolution. At each spatial position, it forms each output channel as a weighted combination of the input channels. It uses stride 2 when the block halves the spatial dimensions. [Paper, Section 3.3](https://arxiv.org/pdf/1512.03385#page=4)

| Option | When the shape stays the same | When the shape changes |
| --- | --- | --- |
| **A** | Identity | Stride-2 subsampling and zero-padding of channels |
| **B** | Identity | Learned projection with stride 2 |
| **C** | Learned projection with stride 1 | Learned projection with stride 2 |

The authors compared all three on ImageNet with ResNet-34. The accuracy gains from A to B to C were small, with more shortcut parameters in each step. For the CIFAR-10 ResNets in Section 4.2, the authors used **option A**, which is what we implement below. [Paper, Section 4.1](https://arxiv.org/pdf/1512.03385#page=6) · [Section 4.2](https://arxiv.org/pdf/1512.03385#page=7)

### The shortcut at a stage boundary

At the start of stages 2 and 3, the first convolution uses stride 2 to halve the spatial dimensions and doubles the channel count. The **option A shortcut** matches that shape with two fixed operations. At the first stage boundary, the shortcut starts with $16\times32\times32$ features:

1. Take every other row and column with the strided slice to get $16\times16\times16$.
2. Pad the channel dimension with eight zero channels on each side to get $32\times16\times16$.

Both branches now have the same shape and can be added element by element. The shortcut still has no learned parameters. The paper specifies zero-padding at this step. The code below places those zeros symmetrically around the original channels. [Paper, Section 3.3](https://arxiv.org/pdf/1512.03385#page=4) · [CIFAR-10 architecture, Section 4.2](https://arxiv.org/pdf/1512.03385#page=7)

In [1]:
import torch
from torch import nn
from torch.nn import functional as F


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.stride = stride
        self.channel_padding = (out_channels - in_channels) // 2
        self.residual = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        shortcut = x[:, :, ::self.stride, ::self.stride]
        if self.channel_padding:
            pad = self.channel_padding
            shortcut = F.pad(shortcut, (0, 0, 0, 0, pad, pad))
        return F.relu(self.residual(x) + shortcut)

## 3. The paper’s CIFAR-10 ResNet-56

Section 4.2 defines a family of CIFAR-10 models with $6n+2$ weighted layers, where $n$ is the number of residual blocks per stage. ResNet-56 uses $n=9$. The stem is a $3\times3$ convolution with stride 1, and the two stage transitions use stride 2 and option A shortcuts. Shapes below omit the batch dimension.

| Part | Output shape | Weighted layers |
| --- | --- | ---: |
| Stem convolution | $16\times32\times32$ | 1 |
| Stage 1 | $16\times32\times32$ | $9\times2$ |
| Stage 2 | $32\times16\times16$ | $9\times2$ |
| Stage 3 | $64\times8\times8$ | $9\times2$ |
| Global average pooling | 64 features | 0 |
| Linear classifier | 10 scores | 1 |

The **56** counts the 55 convolutions and the final linear layer. Global average pooling averages each $8\times8$ feature map to a single value before classification. Table 6 reports about 0.85 million parameters for this model. [Paper, Section 4.2 and Table 6](https://arxiv.org/pdf/1512.03385#page=7)

The paper describes the ten-way classifier followed by softmax. This PyTorch model returns the ten logits so the training notebook can pass them directly to `CrossEntropyLoss`, which includes log-softmax. When we want probabilities, we can apply `logits.softmax(dim=1)`. [PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.modules.loss.CrossEntropyLoss.html)

In [2]:
def make_stage(in_channels, out_channels, stride):
    blocks = [ResidualBlock(in_channels, out_channels, stride)]
    blocks.extend(ResidualBlock(out_channels, out_channels) for _ in range(8))
    return nn.Sequential(*blocks)


model = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1, bias=False),
    nn.BatchNorm2d(16),
    nn.ReLU(),
    make_stage(16, 16, stride=1),
    make_stage(16, 32, stride=2),
    make_stage(32, 64, stride=2),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(64, 10),
)

for layer in model.modules():
    if isinstance(layer, nn.Conv2d):
        nn.init.kaiming_normal_(layer.weight, mode="fan_in", nonlinearity="relu")

## 4. A first forward pass

A batch of two example images should produce a tensor of shape `(2, 10)`. Each row contains the ten raw class scores, or logits, for one image.

In [3]:
model.eval()
with torch.inference_mode():
    logits = model(torch.zeros(2, 3, 32, 32))

weighted_layers = sum(isinstance(layer, (nn.Conv2d, nn.Linear)) for layer in model.modules())
parameters = sum(parameter.numel() for parameter in model.parameters())

print(f"Output shape: {tuple(logits.shape)}")
print(f"Convolutions and linear layers: {weighted_layers}")
print(f"Parameters: {parameters:,}")

Output shape: (2, 10)
Convolutions and linear layers: 56
Parameters: 853,018


## 5. What the CIFAR-10 experiments show

In Figure 6, the authors compare plain and residual networks of the same depth and width. Option A adds zero learned parameters, so each pair also has the same parameter count. The deeper plain models have higher training error, while the residual versions train more successfully. The matched comparison links the improvement to the shortcut design. [Paper, Section 4.2 and Figure 6](https://arxiv.org/pdf/1512.03385#page=8)

Table 6 gives **6.97% test error** for ResNet-56, the reference for our evaluation notebook. The paper also reports that ResNet-1202 reaches below 0.1% training error yet has 7.93% test error, compared with the best reported 6.43% for ResNet-110. That result separates the ability to fit the training set from the ability to generalize. [Table 6](https://arxiv.org/pdf/1512.03385#page=7) · [Analysis of ResNet-1202](https://arxiv.org/pdf/1512.03385#page=8)